# 13 · Final stacker: finer tagger features, a second tagger, tuned gradient boosting

The reference is the stacker of notebook 11 (0.967 out-of-fold, 0.968 public). Its remaining fusion errors are mostly
texts with a very short AI section, so three changes are tried, each against that reference and then combined:

1. **Finer summaries of the first tagger.** From the token probabilities saved in notebook 12: the highest AI
   probability smoothed over 8 and 16 tokens (instead of only 32), the number of tokens above 0.9, the mean of the ten
   most AI-looking tokens, and sentence-level means (highest sentence mean, number and share of sentences with mean
   above 0.5).
2. **A second tagger.** The token probabilities of both taggers are averaged token by token before summarizing.
3. **Gradient-boosting settings.** A small grid (more trees with a smaller learning rate, larger trees) for the best
   feature set, then **bagging**: five copies, each seeing a random half of the features at every split, averaged.

All scores are out-of-fold on the same 5 group folds inside validation; the reference is read from notebook 11.

In [ ]:
# Setup: clone/update the repo on Colab/Kaggle, install requirements, resolve paths, get the data
import os, sys, subprocess
REPO_URL = "https://github.com/anna-kryvulia/ucu-ai-hw-generated-text-detection.git"
BRANCH = "main"   # branch with the code you pushed
if "google.colab" in sys.modules or os.path.exists("/kaggle/working"):
    ROOT = "/content/repo" if "google.colab" in sys.modules else "/kaggle/working/repo"
    if not os.path.exists(ROOT):
        subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, ROOT], check=True)
    else:
        subprocess.run(["git", "-C", ROOT, "pull", "-q"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{ROOT}/requirements.txt"], check=True)
else:
    ROOT = os.path.abspath("..")
sys.path.insert(0, f"{ROOT}/src")

from hwkit import env
from hwkit.download import ensure_data
from hwkit.tracker import Run
P = env.setup(seed=42)
ensure_data(P)

In [ ]:
# Imports, config; validation and test texts; feature blocks of notebook 11 and the new tagger blocks
import json
import time
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import f1_score
from sklearn.preprocessing import OneHotEncoder

from hwkit.config import load_config
from hwkit.split import LABELS, load_split, scores
from hwkit.stack import chunk_features, group_folds, out_of_fold, text_features
from hwkit.submission import make_submission
from hwkit.tagger import load_tokens, tag_features

NOTEBOOK_START = time.time()
cfg = load_config("stack_final")
METRICS = ["macro_f1_testlike", "macro_f1", "accuracy", "f1_human", "f1_fusion", "f1_ai"]
OLD_TAG = ["tag_mean", "tag_share", "tag_std", "tag_runs", "tag_longest_share", "tag_longest_log",
           "tag_smooth_max", "tag_smooth_min"]                     # the eight features of notebook 10

val = load_split(pd.read_csv(P.data / "train.csv", usecols=["id", "text", "label", "domain", "lang"]))
val = val[val["fold"] == 0].reset_index(drop=True)
test = pd.read_csv(P.data / "test.csv")
for df in (val, test):
    df["domain_lang"] = df["domain"] + " | " + df["lang"]
y = val["label"].map({c: i for i, c in enumerate(LABELS)}).to_numpy()
w = val["weight"].to_numpy()
folds = group_folds(val, cfg["cv"]["folds"])
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False).fit(val[["domain_lang"]])

def run_features(run_id, split, ids):
    """Chunk features of a saved encoder run, rows in the order of `ids`."""
    pred_dir = P.runs / run_id / "predictions"
    order = pd.read_csv(pred_dir / f"{split}.csv", usecols=["id"])["id"]
    f = chunk_features(dict(np.load(pred_dir / f"chunks_{split}.npz")), len(order), run_id)
    return f.set_index(order).loc[ids].reset_index(drop=True)

def token_features(file, df):
    """Per-text summaries (old and fine) of saved token probabilities, rows in the order of df."""
    ids, probs, starts = load_tokens(P.runs / cfg["inputs"]["tagger_tokens"] / "predictions" / file)
    text_of = df.set_index("id")["text"]
    f = pd.DataFrame([tag_features(p, s, text_of[i], cfg["features"]["windows"], fine=True)
                      for i, p, s in zip(ids, probs, starts)], index=ids)
    return f.loc[df["id"]].reset_index(drop=True)

def average_features(split, df):
    """Old and fine summaries of the token-wise average of both taggers' probabilities."""
    d = P.runs / cfg["inputs"]["tagger_tokens"] / "predictions"
    ids, p1, starts = load_tokens(d / f"tokens_tagger1_{split}.npz")
    ids2, p2, _ = load_tokens(d / f"tokens_{split}.npz")
    assert (ids == ids2).all() and all(len(a) == len(b) for a, b in zip(p1, p2))
    text_of = df.set_index("id")["text"]
    f = pd.DataFrame([tag_features((a + b) / 2, s, text_of[i], cfg["features"]["windows"], fine=True)
                      for i, a, b, s in zip(ids, p1, p2, starts)], index=ids)
    return f.loc[df["id"]].reset_index(drop=True).add_prefix("avg_")

def blocks_for(split, df):
    tagger = pd.read_csv(P.runs / cfg["inputs"]["tagger"] / "predictions" / f"features_{split}.csv").set_index("id")
    tfidf = pd.read_csv(P.runs / cfg["inputs"]["baseline"] / "predictions" / f"{split}.csv").set_index("id")
    enc = pd.concat([run_features(r, split, df["id"]) for r in cfg["inputs"]["encoders"]], axis=1)
    base_cols = [c for c in enc.columns if "_mean_" in c or "_first_" in c or c.endswith("_log_chunks")]
    first = token_features(f"tokens_tagger1_{split}.npz", df)
    return {
        "base": pd.concat([enc[base_cols],
                           np.log(np.clip(tfidf.loc[df["id"], [f"s_{c}" for c in LABELS]], 1e-6, 1))
                           .reset_index(drop=True).add_prefix("tfidf_"),
                           pd.DataFrame(ohe.transform(df[["domain_lang"]]), columns=ohe.categories_[0])], axis=1),
        "shape": enc.drop(columns=base_cols),
        "text": text_features(df["text"]),
        "tagger": tagger.loc[df["id"]].reset_index(drop=True),
        "fine1": first.drop(columns=OLD_TAG),
        "avg": average_features(split, df),
    }

t0 = time.time()
B = {"val": blocks_for("val", val), "test": blocks_for("test", test)}
run = Run(cfg["name"], cfg, P)
print({k: v.shape[1] for k, v in B["val"].items()}, f"features; {(time.time() - t0) / 60:.1f} min")

In [ ]:
# New features: 10th, 50th and 90th percentile per class on validation (unweighted)
B["val"]["fine1"].assign(label=val["label"]).groupby("label").quantile([0.1, 0.5, 0.9]).unstack().T.unstack()[LABELS].round(3)

## Feature variants, each against the reference

In [ ]:
# Out-of-fold scores of each feature set with the gradient boosting of notebook 11; refit on all of validation for test
class Bagged:
    """Several boosting models that each see a random share of the features at every split; probabilities averaged."""
    def __init__(self, copies, max_features, **params):
        self.models = [HistGradientBoostingClassifier(**params, max_features=max_features, early_stopping=False,
                                                      random_state=seed) for seed in range(copies)]
    def fit(self, X, y, sample_weight=None):
        for m in self.models:
            m.fit(X, y, sample_weight=sample_weight)
        return self
    def predict_proba(self, X):
        return np.mean([m.predict_proba(X) for m in self.models], axis=0)

def boosting(**over):
    return lambda: HistGradientBoostingClassifier(**{**cfg["boosting"], **over}, early_stopping=False, random_state=42)

rows, oof, test_prob = [], {}, {}
def evaluate(name, blocks, make):
    t0 = time.time()
    Xv = pd.concat([B["val"][b] for b in blocks], axis=1).to_numpy()
    Xt = pd.concat([B["test"][b] for b in blocks], axis=1).to_numpy()
    oof[name] = out_of_fold(lambda r: make().fit(Xv[r], y[r], sample_weight=w[r]),
                            lambda m, r: m.predict_proba(Xv[r]), folds)
    test_prob[name] = make().fit(Xv, y, sample_weight=w).predict_proba(Xt)
    res = scores(val["label"], np.array(LABELS)[oof[name].argmax(axis=1)], val["weight"])
    run.log_metrics(name, res)
    rows.append({"variant": name, **{k: res[k] for k in METRICS}, "minutes": (time.time() - t0) / 60})
    return res["macro_f1_testlike"]

def table():
    ref = json.loads((P.runs / "stack_tagger" / "metrics.json").read_text())["all features + tagger"]
    t = pd.DataFrame([{"variant": "reference: notebook 11", **{k: ref[k] for k in METRICS}}] + rows)
    t.insert(1, "vs reference", t["macro_f1_testlike"] - t.loc[0, "macro_f1_testlike"])
    return t.round(4)

OTHER = ["base", "shape", "text"]
FEATURE_SETS = {
    "+ finer features, first tagger": OTHER + ["tagger", "fine1"],
    "two taggers averaged (all summaries)": OTHER + ["avg"],
    "+ finer features and two taggers": OTHER + ["tagger", "fine1", "avg"],
}
for name, blocks in FEATURE_SETS.items():
    evaluate(name, blocks, boosting())
table()

## Boosting settings and bagging for the best feature set

In [ ]:
# Grid of boosting settings on the best feature set, then bagging of the best setting
best_set = max(FEATURE_SETS, key=lambda n: next(r["macro_f1_testlike"] for r in rows if r["variant"] == n))
grid = {}
for g in cfg["boosting_grid"]:
    name = f"{best_set} | trees {g['max_iter']}, lr {g['learning_rate']}, leaves {g['max_leaf_nodes']}"
    grid[name] = (g, evaluate(name, FEATURE_SETS[best_set], boosting(**g)))
best_name, (best_g, _) = max(grid.items(), key=lambda kv: kv[1][1])
evaluate(f"{best_name} | bagged x{cfg['bagging']['copies']}", FEATURE_SETS[best_set],
         lambda: Bagged(**cfg["bagging"], **{**cfg["boosting"], **best_g}))
summary = table()
summary

In [ ]:
# Macro-F1 per domain (over the classes present there): reference (saved out-of-fold predictions) vs the best variant
best = summary.iloc[1:].sort_values("macro_f1_testlike").iloc[-1]["variant"]
cols = [f"s_{c}" for c in LABELS]
oof["reference"] = pd.read_csv(P.runs / "stack_tagger" / "predictions" / "val_oof.csv").set_index("id").loc[val["id"], cols].to_numpy()
pred = {name: np.array(LABELS)[oof[name].argmax(axis=1)] for name in ("reference", best)}
dom = pd.DataFrame({name: {dl: f1_score(g["label"], pred[name][g.index], labels=sorted(g["label"].unique()),
                                        average="macro", zero_division=0)
                           for dl, g in val.groupby("domain_lang")} for name in pred})
dom.columns = ["reference", "best"]
dom["texts"] = val["domain_lang"].value_counts()
dom["change"] = dom["best"] - dom["reference"]
print("best variant:", best)
dom.round(3).sort_values("change")

In [ ]:
# Fusion recall by the share of AI-written words (notebook 08): did the short AI sections get easier?
share = pd.read_csv(P.repo / "results" / "fusion_spans.csv.gz").set_index("id")["ai_share"]
fus = val[val["label"] == "fusion"].assign(ai_share=lambda d: d["id"].map(share))
fus["bucket"] = pd.cut(fus["ai_share"], [0, 0.05, 0.1, 0.2, 1.0])
pd.DataFrame({name: (pred[name][fus.index] == "fusion") for name in pred}, index=fus.index).groupby(
    fus["bucket"], observed=True).mean().assign(texts=fus["bucket"].value_counts()).round(3)

In [ ]:
# Confusion matrix of the best variant on validation (out-of-fold, rows = true label, counts)
pd.crosstab(val["label"], pd.Series(pred[best], name="predicted"))[LABELS].loc[LABELS]

In [ ]:
# Save out-of-fold and test probabilities of the best variant; write its submission file
pd.concat([val[["id", "label"]], pd.DataFrame(oof[best], columns=cols)], axis=1).to_csv(run.pred_path("val_oof"), index=False)
pd.concat([test[["id"]], pd.DataFrame(test_prob[best], columns=cols)], axis=1).to_csv(run.pred_path("test"), index=False)
sub = make_submission(test["id"], np.array(LABELS)[test_prob[best].argmax(axis=1)],
                      P.repo / "submissions" / f"{cfg['name']}.csv", P.data)

In [ ]:
# Distinct warnings logged in this session (should be empty), total time
print(f"total: {(time.time() - NOTEBOOK_START) / 60:.1f} min")
env.warnings_summary(P)